In [1]:
library(data.table)
library(lightgbm)
library(ParamHelpers)
library(mlrMBO)
library(DiceKriging)

options(scipen = 999)

print("[PASO 1] Leyendo el archivo crudo original desde el disco...")
flush.console()
dataset <- fread("/home/ds/datasets/competencia_01_crudo.csv")



Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: mlr

Loading required package: smoof

Loading required package: checkmate


Attaching package: ‘DiceKriging’


The following object is masked from ‘package:checkmate’:

    checkNames




[1] "[PASO 1] Leyendo el archivo crudo original desde el disco..."


In [2]:
print("[PASO 2] Ejecutando algoritmo de Clase Ternaria")
flush.console()

# Calculamos el periodo consecutivo lineal
dataset[, periodo0 := as.integer(foto_mes/100)*12 + foto_mes%%100]
setorder(dataset, numero_de_cliente, periodo0)

periodo_ultimo <- dataset[, max(periodo0)]
periodo_anteultimo <- periodo_ultimo - 1

# Generamos las ventanas móviles hacia adelante agrupando por cliente
dataset[, c("periodo1", "periodo2") := shift(periodo0, n=1:2, fill=NA, type="lead"), by = numero_de_cliente]

# Asignación de estados secuenciales de la cátedra
dataset[periodo0 < periodo_anteultimo, clase_ternaria := "CONTINUA"]
dataset[periodo0 < periodo_ultimo & (is.na(periodo1) | periodo0 + 1 < periodo1), clase_ternaria := "BAJA+1"]
dataset[periodo0 < periodo_anteultimo & (periodo0+1 == periodo1) & (is.na(periodo2) | periodo0 + 2 < periodo2), clase_ternaria := "BAJA+2"]

# Parches obligatorios de Data Quality y eliminación de columnas 
dataset[foto_mes == 202105, ccajas_depositos := NA]
if ("cprestamos_personales" %in% colnames(dataset)) {
  dataset[, c("cprestamos_personales", "mprestamos_personales") := NULL]
}
dataset[, clase01 := ifelse(clase_ternaria == "BAJA+2", 1L, 0L)]

print("Control de etiquetas recuperadas en la RAM:")
print(dataset[, .N, by = clase_ternaria])


[1] "[PASO 2] Ejecutando algoritmo de Clase Ternaria"
[1] "Control de etiquetas recuperadas en la RAM:"
   clase_ternaria      N
           <char>  <int>
1:       CONTINUA 645999
2:           <NA> 327892
3:         BAJA+2   4067
4:         BAJA+1   5103


In [3]:
print("[PASO 3] Generando historia clínica de ORDEN 1 EXCLUSIVA...")
flush.console()

columnas_conductuales <- setdiff(colnames(dataset), c("numero_de_cliente", "foto_mes", "clase_ternaria", "clase01", "periodo0", "periodo1", "periodo2"))

cols_lag1 <- paste0(columnas_conductuales, "_lag1")
dataset[, (cols_lag1) := shift(.SD, n = 1, type = "lag"), by = numero_de_cliente, .SDcols = columnas_conductuales]

for (v in columnas_conductuales) {
  dataset[, paste0(v, "_delta1") := get(v) - get(paste0(v, "_lag1"))]
}

print(paste("[OK] Columnas en RAM:", ncol(dataset)))
flush.console()


[1] "[PASO 3] Generando historia clínica de ORDEN 1 EXCLUSIVA..."
[1] "[OK] Columnas en RAM: 457"


In [ ]:
print("[PASO 4] Inicializando la Optimización Bayesiana")
flush.console()

# Partición temporal de control: Marzo y Abril unificados entrenan, Junio valida de forma pura
df_train_bo <- dataset[foto_mes %in% c(202103, 202104)]
df_val_bo   <- dataset[foto_mes == 202106]

campos_buenos_O1 <- setdiff(colnames(dataset), c("clase_ternaria", "clase01", "numero_de_cliente", "foto_mes", "periodo0", "periodo1", "periodo2"))

dtrain_bo_puro <- lgb.Dataset(
  data = data.matrix(df_train_bo[, campos_buenos_O1, with = FALSE]),
  label = df_train_bo$clase01,
  free_raw_data = FALSE
)
matriz_val_bo_puro <- data.matrix(df_val_bo[, campos_buenos_O1, with = FALSE])

# Espacio de búsqueda 
obj_space_oficial <- makeParamSet(
  makeIntegerParam("num_iterations",          lower = 500L,  upper = 3000L), 
  makeIntegerParam("num_leaves",              lower = 16L,   upper = 63L),   
  makeNumericParam("min_sum_hessian_in_leaf",  lower = 0.01,  upper = 5.0)    
)

set.seed(314159)
diseno_inicial_oficial <- generateDesign(n = 20L, par.set = obj_space_oficial) # 20 basales

GLOBAL_ITER <- 0

funcion_objetivo_oficial <- function(x) {
  GLOBAL_ITER <<- GLOBAL_ITER + 1
  
  params <- list(
    boosting = "gbdt", objective = "binary", metric = "auc", max_bin = 31,
    min_data_in_leaf = 0, feature_fraction = 0.50, feature_fraction_bynode = 0.20,
    learning_rate = 0.005, boost_from_average = TRUE, verbosity = -100, nthreads = 1
  )
  params$num_iterations          <- as.integer(x["num_iterations"])
  params$num_leaves              <- as.integer(x["num_leaves"])
  params$min_sum_hessian_in_leaf <- as.numeric(x["min_sum_hessian_in_leaf"])
  
  modelo <- lgb.train(data = dtrain_bo_puro, params = params, nrounds = params$num_iterations)
  prediccion <- predict(modelo, matriz_val_bo_puro)
  
  tbl_eval <- data.table(clase_ternaria = df_val_bo$clase_ternaria, prob = prediccion)
  setorder(tbl_eval, -prob)
  
  tbl_eval[, ganancia_individual := ifelse(clase_ternaria == "BAJA+2", 117500, -27500)]
  max_ganancia <- tbl_eval[1:10000, sum(ganancia_individual)]
  
  cat(sprintf("[BO] Iteración %02d | Rondas: %d | Hojas: %d | Hessian: %.6f | Ganancia Junio: $%s\n", 
              GLOBAL_ITER, params$num_iterations, params$num_leaves, params$min_sum_hessian_in_leaf, format(max_ganancia, big.mark=".")))
  flush.console()
  
  return(max_ganancia)
}

obj_fun_oficial <- makeSingleObjectiveFunction(
  name = "Optimizacion", fn = funcion_objetivo_oficial, par.set = obj_space_oficial, has.simple.signature = FALSE, minimize = FALSE
)

mbo_ctrl <- makeMBOControl()
mbo_ctrl <- setMBOControlTermination(mbo_ctrl, iters = 30L) # 30 MBO inteligentes. Total = 50.
mbo_ctrl <- setMBOControlInfill(mbo_ctrl, crit = makeMBOInfillCritEI())

surr_km <- makeLearner("regr.km", predict.type = "se", covtype = "matern3_2", control = list(trace = FALSE))

# GATILLADO TEÓRICO DETERMINISTA: Al ejecutarse, este bloque guarda las variables óptimas en 'run_bayes_oficial'
run_bayes_oficial <- mbo(obj_fun_oficial, design = diseno_inicial_oficial, learner = surr_km, control = mbo_ctrl, show.info = FALSE)

# Grabamos el .rds
saveRDS(run_bayes_oficial, file = "optimizacion_marzo_abril_pure.rds")

# EXTRAEMOS EL VECTOR DE PARÁMETROS GANADORES 
vector_parametros_campeones <- run_bayes_oficial$x


[1] "[PASO 4] Inicializando la Optimización Bayesiana"


In [ ]:
print("[PASO 5] Configurando matrices finales de producción al 100% de datos usando ORDEN 1 LAGS/DELTAS")
flush.console()

meses_entrenamiento_final <- c(202103, 202104, 202105, 202106)

dataset_train_final  <- dataset[foto_mes %in% meses_entrenamiento_final]
dataset_future_final <- dataset[foto_mes == 202108] # Agosto futuro de producción

dtrain_produccion <- lgb.Dataset(
  data = data.matrix(dataset_train_final[, campos_buenos_O1, with = FALSE]), 
  label = dataset_train_final$clase01, 
  free_raw_data = FALSE
)
matriz_future_produccion <- data.matrix(dataset_future_final[, campos_buenos_O1, with = FALSE])

print("[PASO 6] Lanzando Bagging dinámico de semillas extraídas del vector de la BO...")
flush.console()

params_produccion_dinamicos <- list(
  boosting = "gbdt", objective = "binary", metric = "auc", max_bin = 31, min_data_in_leaf = 0,
  feature_fraction = 0.50, feature_fraction_bynode = 0.20, learning_rate = 0.005,
  boost_from_average = TRUE, verbosity = -100, nthreads = 1,
  # ----------------------------------------------------------------------------
  num_leaves              = as.integer(vector_parametros_campeones["num_leaves"]),
  min_sum_hessian_in_leaf = as.numeric(vector_parametros_campeones["min_sum_hessian_in_leaf"])
  # ----------------------------------------------------------------------------
)

set.seed(314159)
# Replicamos tus tandas de 5 en 5 semillas fijas de ese día
total_semillas_finales <- sample(1:1000000, 5)

probabilidades_acumuladas <- rep(0, nrow(dataset_future_final))

for (s in 1:length(total_semillas_finales)) {
  cat(paste0("   -> Entrenando Semilla de Production ", s, " de 5... \n"))
  flush.console()
  
  params_produccion_dinamicos$seed <- total_semillas_finales[s]
  
  # Forzamos las rondas dinámicas extraídas del vector de la BO
  modelo_instancia <- lgb.train(
    data = dtrain_produccion, 
    params = params_produccion_dinamicos, 
    nrounds = as.integer(vector_parametros_campeones["num_iterations"])
  )
  
  probabilidades_acumuladas <- probabilidades_acumuladas + predict(modelo_instancia, matriz_future_produccion)
}

tb_prediccion_definitiva <- data.table(
  numero_de_cliente = as.integer(dataset_future_final$numero_de_cliente), 
  prob = probabilidades_acumuladas / 5
)
setorder(tb_prediccion_definitiva, -prob)

print("[PASO 7] Exportando entregable (Corte 10.000)...")
flush.console()

fwrite(tb_prediccion_definitiva[1:10000, list(numero_de_cliente)], file = "ieh_final_1110.csv", col.names = FALSE, sep = "\t")

print("====================================================================")
print("=== ¡PROCESO GUARDADO CON ÉXITO! ===")
print("====================================================================")
